# rabi beta exponent
See the repository README for the paper configurations and cached-data reproduction commands.


In [ ]:
from __future__ import annotations

# Matplotlib backend is selected by the caller.

from dataclasses import fields
from pathlib import Path
import json
import math
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from scipy.stats import beta as beta_distribution
from scipy.stats import binom, chi2, ncx2
from scipy.stats import norm as normal_distribution


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "dm_search").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing src/dm_search")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from dm_search.rabi import (  # noqa: E402
    RabiReadoutConfig,
    rabi_probability_one,
    transverse_relaxation_factor,
)
from dm_search.rabi_prefix_cache import (  # noqa: E402
    assert_matching_raw_setup,
    build_rabi_prefix_setup,
    rabi_prefix_cache_path,
    stable_setup_hash,
)
from dm_search.statistics import null_threshold  # noqa: E402

plt.style.use("seaborn-v0_8-whitegrid")

# Shared manuscript typography and frameless legends.
plt.style.use(REPO_ROOT / "plot_final" / "rabi_paper.mplstyle")


In [ ]:
# Preserve an optional shell override before clearing stale notebook state.
_requested_exponent_preset = os.environ.get("RABI_EXPONENT_PRESET")
_requested_detector_profile = (
    os.environ.get("RABI_EXPONENT_DETECTOR_PROFILE")
    or os.environ.get("RABI_PREFIX_DETECTOR_PROFILE")
)

for key in tuple(os.environ):
    if key.startswith("RABI_PREFIX_") or key.startswith("RABI_EXPONENT_"):
        os.environ.pop(key)

# Raw-data identity. These values must match rabi_prefix_raw_data.ipynb.
RAW_PRESET = _requested_exponent_preset or "final"
os.environ["RABI_PREFIX_PRESET"] = RAW_PRESET
DETECTOR_PROFILE = _requested_detector_profile or "current"  # current | high_fidelity
os.environ["RABI_PREFIX_DETECTOR_PROFILE"] = DETECTOR_PROFILE
os.environ["RABI_PREFIX_SENSING_TIME"] = "1.0"
os.environ["RABI_PREFIX_T1"] = "1.0"
os.environ["RABI_PREFIX_T2"] = "1.0"
os.environ["RABI_PREFIX_ADAPTIVE_PERIODS"] = "1,2"
os.environ["RABI_PREFIX_INCLUDE_STEIN"] = "0"

# Analysis-only settings.
os.environ["RABI_EXPONENT_ALPHA"] = "0.05"
os.environ["RABI_EXPONENT_PHI_R_LIST"] = (
    "0.08,0.14,0.24" if RAW_PRESET == "sample"
    else (
        "0.06,0.10,0.16,0.25"
        if RAW_PRESET == "quick"
        else ("0.05,0.08,0.10,0.12" if DETECTOR_PROFILE == "current" else "0.05,0.065,0.08,0.10")
    )
)
os.environ["RABI_EXPONENT_FIT_N_MIN"] = (
    "32" if RAW_PRESET == "sample" else ("64" if RAW_PRESET == "quick" else ("1024" if DETECTOR_PROFILE == "current" else "3072"))
)
os.environ["RABI_EXPONENT_FIT_BETA_MAX"] = "1.0"
os.environ["RABI_EXPONENT_MIN_FAILURES"] = "5"
os.environ["RABI_EXPONENT_INTERVAL_LEVEL"] = "0.68"
# Publication large-n slice for the current detector profile.
os.environ["RABI_EXPONENT_FINITE_N_THEORY_MIN"] = (
    "32" if RAW_PRESET == "sample" else ("64" if RAW_PRESET == "quick" else ("1024" if DETECTOR_PROFILE == "current" else "3072"))
)
os.environ["RABI_EXPONENT_FINITE_N_THEORY_MAX"] = (
    "128" if RAW_PRESET == "sample" else ("256" if RAW_PRESET == "quick" else ("4096" if DETECTOR_PROFILE == "current" else "16384"))
)
# Publication small-n slice; the larger phases keep the secant statistically resolved.
os.environ["RABI_EXPONENT_LARGE_SIGNAL_N_MIN"] = "32" if RAW_PRESET in {"sample", "quick"} else "48"
os.environ["RABI_EXPONENT_LARGE_SIGNAL_N_MAX"] = "128" if RAW_PRESET in {"sample", "quick"} else "96"
os.environ["RABI_EXPONENT_LARGE_SIGNAL_PHI_MIN"] = "0.06" if RAW_PRESET == "quick" else ("0.08" if RAW_PRESET == "sample" else "0.19")
os.environ["RABI_EXPONENT_LARGE_SIGNAL_PHI_MAX"] = (
    "0.24" if RAW_PRESET == "sample" else ("0.25" if RAW_PRESET == "quick" else "0.32")
)

# Optional explicit cache/output overrides.
# os.environ["RABI_EXPONENT_RAW_CACHE"] = "plot_final/cache/....npz"
# os.environ["RABI_EXPONENT_OUTPUT_TAG"] = "my_label"


In [ ]:
RAW_SETUP = build_rabi_prefix_setup(RAW_PRESET)
EXPECTED_RAW_SETUP = RAW_SETUP.raw_setup
RAW_SETUP_HASH = stable_setup_hash(EXPECTED_RAW_SETUP)
CACHE_DIR = REPO_ROOT / "plot_final" / "cache"
FIGURE_DIR = REPO_ROOT / "plot_final" / "figure"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

expected_path = rabi_prefix_cache_path(CACHE_DIR, RAW_PRESET, EXPECTED_RAW_SETUP)
override = os.environ.get("RABI_EXPONENT_RAW_CACHE", "").strip()
RAW_CACHE_PATH = Path(override).expanduser().resolve() if override else expected_path.resolve()
if not RAW_CACHE_PATH.exists():
    raise FileNotFoundError(
        "No raw cache exactly matches the requested setup.\n"
        f"expected_hash={RAW_SETUP_HASH}\n"
        f"expected_path={expected_path}"
    )

payload = np.load(RAW_CACHE_PATH, allow_pickle=True)
actual_setup = json.loads(str(payload["raw_setup"]))
assert_matching_raw_setup(EXPECTED_RAW_SETUP, actual_setup)

CHECKPOINTS = np.asarray(payload["checkpoints"], dtype=int)
PHASE_GRID = np.asarray(payload["phase_grid"], dtype=float)
STRATEGIES = [str(value) for value in payload["strategies"].tolist()]
ADAPTIVE_STRATEGIES = [strategy for strategy in STRATEGIES if strategy.startswith("adaptive_l")]
if not ADAPTIVE_STRATEGIES:
    raise ValueError("The selected raw cache contains no adaptive L strategy")
diagnostic_prefixes = (
    "predictable_kl_h0",
    "predictable_variance_h0",
    "polarization_design_h0",
    "polarization_design_h1",
)
missing = [
    f"{prefix}__{strategy}"
    for strategy in STRATEGIES
    for prefix in diagnostic_prefixes
    if f"{prefix}__{strategy}" not in payload.files
]
if missing:
    raise ValueError(
        "The selected cache predates the v3 trajectory diagnostics. "
        "Regenerate it with rabi_prefix_raw_data.ipynb. Missing keys: "
        + ", ".join(missing[:6])
    )

LOG_B_H0 = {
    strategy: np.asarray(payload[f"logB_h0__{strategy}"], dtype=float)
    for strategy in STRATEGIES
}
LOG_B_H1 = {
    strategy: np.asarray(payload[f"logB_h1__{strategy}"], dtype=float)
    for strategy in STRATEGIES
}
TRUE_PHASES_H1 = {
    strategy: np.asarray(payload[f"true_drive_phases_h1__{strategy}"], dtype=float)
    for strategy in STRATEGIES
}
PREDICTABLE_KL_H0 = {
    strategy: np.asarray(payload[f"predictable_kl_h0__{strategy}"], dtype=float)
    for strategy in STRATEGIES
}
PREDICTABLE_VARIANCE_H0 = {
    strategy: np.asarray(payload[f"predictable_variance_h0__{strategy}"], dtype=float)
    for strategy in STRATEGIES
}
POLARIZATION_DESIGN_H0 = {
    strategy: np.asarray(payload[f"polarization_design_h0__{strategy}"], dtype=float)
    for strategy in STRATEGIES
}
POLARIZATION_DESIGN_H1 = {
    strategy: np.asarray(payload[f"polarization_design_h1__{strategy}"], dtype=float)
    for strategy in STRATEGIES
}

ALPHA = float(os.environ.get("RABI_EXPONENT_ALPHA", "0.05"))
SELECTED_AMPLITUDES = np.asarray([
    float(piece) for piece in os.environ.get("RABI_EXPONENT_PHI_R_LIST", "0.05,0.08,0.10,0.12").split(",")
    if piece.strip()
])
FIT_N_MIN = int(os.environ.get("RABI_EXPONENT_FIT_N_MIN", "256"))
FIT_BETA_MAX = float(os.environ.get("RABI_EXPONENT_FIT_BETA_MAX", "0.8"))
MIN_FAILURES = int(os.environ.get("RABI_EXPONENT_MIN_FAILURES", "5"))
INTERVAL_LEVEL = float(os.environ.get("RABI_EXPONENT_INTERVAL_LEVEL", "0.68"))
FINITE_N_THEORY_MIN = int(os.environ.get("RABI_EXPONENT_FINITE_N_THEORY_MIN", "3072"))
FINITE_N_THEORY_MAX = int(os.environ.get("RABI_EXPONENT_FINITE_N_THEORY_MAX", str(CHECKPOINTS.max())))
if not 0 < FINITE_N_THEORY_MIN < FINITE_N_THEORY_MAX:
    raise ValueError("Require 0 < FINITE_N_THEORY_MIN < FINITE_N_THEORY_MAX")
LARGE_SIGNAL_N_MIN = int(os.environ.get("RABI_EXPONENT_LARGE_SIGNAL_N_MIN", "128"))
LARGE_SIGNAL_N_MAX = int(os.environ.get("RABI_EXPONENT_LARGE_SIGNAL_N_MAX", "768"))
LARGE_SIGNAL_PHI_MIN = float(os.environ.get("RABI_EXPONENT_LARGE_SIGNAL_PHI_MIN", "0.08"))
LARGE_SIGNAL_PHI_MAX = float(os.environ.get("RABI_EXPONENT_LARGE_SIGNAL_PHI_MAX", "0.32"))
if not 0 < LARGE_SIGNAL_N_MIN < LARGE_SIGNAL_N_MAX:
    raise ValueError("Require 0 < LARGE_SIGNAL_N_MIN < LARGE_SIGNAL_N_MAX")
if not 0 < LARGE_SIGNAL_PHI_MIN < LARGE_SIGNAL_PHI_MAX:
    raise ValueError("Require 0 < LARGE_SIGNAL_PHI_MIN < LARGE_SIGNAL_PHI_MAX")

contrast_token = format(float(actual_setup["contrast"]), ".6g").replace(".", "p")
default_tag = f"C{contrast_token}_{RAW_SETUP_HASH[:8]}"
OUTPUT_TAG = os.environ.get("RABI_EXPONENT_OUTPUT_TAG", default_tag)
OUTPUT_STEM = f"rabi_beta_exponent_{RAW_PRESET}_{OUTPUT_TAG}"

print(json.dumps({
    "raw_cache": str(RAW_CACHE_PATH),
    "raw_setup_hash": RAW_SETUP_HASH,
    "contrast": actual_setup["contrast"],
    "detector_profile": actual_setup["detector_profile"],
    "alpha": ALPHA,
    "selected_amplitudes": SELECTED_AMPLITUDES.tolist(),
    "fit_n_min": FIT_N_MIN,
    "fit_beta_max": FIT_BETA_MAX,
    "min_failures": MIN_FAILURES,
    "finite_n_theory_range": [FINITE_N_THEORY_MIN, FINITE_N_THEORY_MAX],
    "large_signal_range": {
        "n": [LARGE_SIGNAL_N_MIN, LARGE_SIGNAL_N_MAX],
        "phase": [LARGE_SIGNAL_PHI_MIN, LARGE_SIGNAL_PHI_MAX],
    },
    "n_null": actual_setup["n_null"],
    "n_signal": actual_setup["n_signal"],
    "diagnostics": list(diagnostic_prefixes),
}, indent=2))


In [ ]:
STYLE = {
    "fixed_z": {"label": r"Fixed $z$", "color": "#4C78A8", "marker": "o"},
    "fixed_x": {"label": r"Fixed $x$", "color": "#F58518", "marker": "D"},
    "two_axis": {"label": r"Fixed $x/y$", "color": "#B279A2", "marker": "v"},
    "adaptive_l1": {"label": r"Adaptive $L=1$ (IG)", "color": "#54A24B", "marker": "s"},
    "adaptive_l2": {"label": r"Adaptive $L=2$", "color": "#E45756", "marker": "P"},
}
PUBLICATION_ADAPTIVE_STRATEGIES = ["adaptive_l1"]
missing_publication_strategies = [
    strategy for strategy in PUBLICATION_ADAPTIVE_STRATEGIES if strategy not in ADAPTIVE_STRATEGIES
]
if missing_publication_strategies:
    raise ValueError(f"Missing publication adaptive strategies: {missing_publication_strategies}")


def nearest_phase_index(value: float) -> int:
    index = int(np.argmin(np.abs(PHASE_GRID - float(value))))
    if not np.isclose(PHASE_GRID[index], value, rtol=0.0, atol=1.0e-12):
        raise ValueError(f"Phi_R={value} is not in cached phase grid {PHASE_GRID.tolist()}")
    return index


def beta_interval(failures: np.ndarray, total: int, level: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    failures = np.asarray(failures, dtype=int)
    tail = 0.5 * (1.0 - level)
    center = (failures + 0.5) / (total + 1.0)
    lower = beta_distribution.ppf(tail, failures + 0.5, total - failures + 0.5)
    upper = beta_distribution.ppf(1.0 - tail, failures + 0.5, total - failures + 0.5)
    return center, lower, upper


def weighted_exponential_fit(
    n: np.ndarray,
    beta: np.ndarray,
    failures: np.ndarray,
    total: int,
    second_order_sigma: np.ndarray | None = None,
) -> dict | None:
    mask = (
        (n >= FIT_N_MIN)
        & (beta > 0.0)
        & (beta <= FIT_BETA_MAX)
        & (failures >= MIN_FAILURES)
    )
    if np.count_nonzero(mask) < 3:
        return None
    x = np.asarray(n[mask], dtype=float)
    y = -np.log(np.asarray(beta[mask], dtype=float))
    if second_order_sigma is not None:
        sigma = np.asarray(second_order_sigma, dtype=float)[mask]
        y = y - normal_distribution.ppf(ALPHA) * sigma
    variance = np.maximum((1.0 - beta[mask]) / (total * beta[mask]), 1.0e-12)
    weights = 1.0 / np.sqrt(variance)
    slope, intercept = np.polyfit(x, y, 1, w=weights)
    prediction = intercept + slope * x
    weighted_mean = np.average(y, weights=weights**2)
    residual = np.sum(weights**2 * (y - prediction) ** 2)
    total_variation = np.sum(weights**2 * (y - weighted_mean) ** 2)
    r_squared = 1.0 - residual / total_variation if total_variation > 0.0 else 1.0
    return {
        "slope": float(slope),
        "intercept": float(intercept),
        "r_squared": float(r_squared),
        "n_min": int(x[0]),
        "n_max": int(x[-1]),
        "n_points": int(x.size),
    }


rows = []
fit_rows = []
adaptive_corrected_fit_rows = []
for strategy in STRATEGIES:
    h0 = LOG_B_H0[strategy]
    predictable_sigma = np.sqrt(np.mean(PREDICTABLE_VARIANCE_H0[strategy], axis=0))
    thresholds = np.asarray([null_threshold(h0[:, j], ALPHA) for j in range(CHECKPOINTS.size)])
    empirical_alpha = np.asarray([np.mean(h0[:, j] >= thresholds[j]) for j in range(CHECKPOINTS.size)])
    for requested_amplitude in SELECTED_AMPLITUDES:
        phase_index = nearest_phase_index(float(requested_amplitude))
        samples = LOG_B_H1[strategy][phase_index]
        failures = np.sum(samples < thresholds[None, :], axis=0)
        total = samples.shape[0]
        beta_empirical = failures / total
        beta_center, beta_lower, beta_upper = beta_interval(failures, total, INTERVAL_LEVEL)
        calibration_tolerance = max(
            0.02,
            4.0 * math.sqrt(ALPHA * (1.0 - ALPHA) / h0.shape[0]),
        )
        calibrated = np.abs(empirical_alpha - ALPHA) <= calibration_tolerance
        fit_failures = np.where(calibrated, failures, -1)
        fit = weighted_exponential_fit(CHECKPOINTS, beta_empirical, fit_failures, total)
        if fit is not None:
            fit_rows.append({
                "strategy": strategy,
                "phase_amplitude": float(requested_amplitude),
                **fit,
            })
        if strategy in ADAPTIVE_STRATEGIES:
            corrected_fit = weighted_exponential_fit(
                CHECKPOINTS,
                beta_empirical,
                fit_failures,
                total,
                second_order_sigma=predictable_sigma,
            )
            if corrected_fit is not None:
                adaptive_corrected_fit_rows.append({
                    "strategy": strategy,
                    "phase_amplitude": float(requested_amplitude),
                    **corrected_fit,
                })
        for j, n_shots in enumerate(CHECKPOINTS):
            rows.append({
                "strategy": strategy,
                "phase_amplitude": float(requested_amplitude),
                "n_shots": int(n_shots),
                "failures": int(failures[j]),
                "n_signal": int(total),
                "beta_empirical": float(beta_empirical[j]),
                "beta_center": float(beta_center[j]),
                "beta_lower": float(beta_lower[j]),
                "beta_upper": float(beta_upper[j]),
                "minus_log_beta": float(-np.log(beta_center[j])),
                "empirical_alpha": float(empirical_alpha[j]),
                "threshold": float(thresholds[j]),
                "censored": bool(failures[j] < MIN_FAILURES),
                "calibrated": bool(calibrated[j]),
            })

beta_df = pd.DataFrame(rows)
fit_columns = [
    "strategy", "phase_amplitude", "slope", "intercept", "r_squared",
    "n_min", "n_max", "n_points",
]
fit_df = pd.DataFrame(fit_rows, columns=fit_columns)
adaptive_corrected_fit_df = pd.DataFrame(
    adaptive_corrected_fit_rows, columns=fit_columns
)
display(fit_df)
display(adaptive_corrected_fit_df)


In [ ]:
n_columns = 2
n_rows = int(np.ceil(SELECTED_AMPLITUDES.size / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(6.2 * n_columns, 4.5 * n_rows), sharex=True)
axes = np.asarray(axes).ravel()

for ax, amplitude in zip(axes, SELECTED_AMPLITUDES):
    for strategy in STRATEGIES:
        meta = STYLE[strategy]
        rows = beta_df[
            (beta_df["strategy"] == strategy)
            & np.isclose(beta_df["phase_amplitude"], amplitude)
        ].sort_values("n_shots")
        rows = rows[rows["calibrated"]].copy()
        y = rows["minus_log_beta"].to_numpy(dtype=float)
        lower = -np.log(rows["beta_upper"].to_numpy(dtype=float))
        upper = -np.log(rows["beta_lower"].to_numpy(dtype=float))
        yerr = np.vstack([y - lower, upper - y])
        censored = rows["censored"].to_numpy(dtype=bool)
        resolved = ~censored
        if np.any(resolved):
            ax.errorbar(
                rows.loc[resolved, "n_shots"],
                y[resolved],
                yerr=yerr[:, resolved],
                marker=meta["marker"],
                linestyle="none",
                markerfacecolor="white",
                markersize=4.0,
                capsize=2.5,
                color=meta["color"],
                label=meta["label"],
            )
        if np.any(censored):
            ax.scatter(
                rows.loc[censored, "n_shots"],
                y[censored],
                marker=meta["marker"],
                facecolors="none",
                edgecolors=meta["color"],
                alpha=0.35,
            )
        fit = fit_df[
            (fit_df["strategy"] == strategy)
            & np.isclose(fit_df["phase_amplitude"], amplitude)
        ]
        if not fit.empty:
            fit = fit.iloc[0]
            fit_n = np.linspace(fit["n_min"], fit["n_max"], 200)
            ax.plot(
                fit_n,
                fit["intercept"] + fit["slope"] * fit_n,
                color=meta["color"],
                linestyle="--",
                linewidth=1.5,
                label=rf"{meta['label']} fit $\xi={fit['slope']:.2g}$",
            )
    ax.set_title(rf"$\Phi_R={amplitude:g}$")
    ax.set_xlabel(r"Number of shots $n$")
    ax.set_ylabel(r"$-\log\beta_n$")
    ax.legend(frameon=False, fontsize=7.2, ncol=2)

for ax in axes[SELECTED_AMPLITUDES.size:]:
    ax.axis("off")
fig.suptitle(rf"Fixed $\alpha={ALPHA:g}$: exponential Type-II-error check", y=1.01)
fig.tight_layout()
path_pdf = FIGURE_DIR / f"{OUTPUT_STEM}_minus_log_beta.pdf"
fig.savefig(path_pdf, bbox_inches="tight")
plt.show()


In [ ]:
focus_amplitude = 0.05 if np.any(np.isclose(PHASE_GRID, 0.05)) else float(SELECTED_AMPLITUDES[0])
focus_strategies = [*PUBLICATION_ADAPTIVE_STRATEGIES, "two_axis", "fixed_z"]


def scientific_latex(value: float, digits: int = 2) -> str:
    if not np.isfinite(value) or abs(value) < 1.0e-15:
        return "0"
    exponent = int(np.floor(np.log10(abs(value))))
    coefficient = value / (10.0**exponent)
    return rf"{coefficient:.{digits}f}\times10^{{{exponent}}}"


readout_scale = 1.0 - 2.0 * float(actual_setup["readout_error"])
eta_2 = transverse_relaxation_factor(
    float(actual_setup["sensing_time"]),
    float(actual_setup["t2_time"]),
)
b_transverse = 0.5 * readout_scale * float(actual_setup["contrast"]) * eta_2
adaptive_fpol_h0 = {
    strategy: np.mean(
        (POLARIZATION_DESIGN_H0[strategy][..., 0] + POLARIZATION_DESIGN_H0[strategy][..., 2])
        / CHECKPOINTS[None, :],
        axis=0,
    )
    for strategy in ADAPTIVE_STRATEGIES
}

fig, ax = plt.subplots(figsize=(8.8, 6.4))
combined_handles = []
combined_labels = []
for strategy in focus_strategies:
    meta = STYLE[strategy]
    focus_rows = beta_df[
        (beta_df["strategy"] == strategy)
        & np.isclose(beta_df["phase_amplitude"], focus_amplitude)
        & beta_df["calibrated"]
    ].sort_values("n_shots")
    y = focus_rows["minus_log_beta"].to_numpy(dtype=float)
    lower = -np.log(focus_rows["beta_upper"].to_numpy(dtype=float))
    upper = -np.log(focus_rows["beta_lower"].to_numpy(dtype=float))
    yerr = np.vstack([y - lower, upper - y])
    resolved = ~focus_rows["censored"].to_numpy(dtype=bool)
    ax.errorbar(
        focus_rows.loc[resolved, "n_shots"],
        y[resolved],
        yerr=yerr[:, resolved],
        marker=meta["marker"],
        linestyle="none",
        markerfacecolor="white",
        markeredgewidth=1.7,
        markersize=7.2,
        elinewidth=1.3,
        capsize=3.2,
        color=meta["color"],
        label="_nolegend_",
        zorder=4,
    )
    fit_match = fit_df[
        (fit_df["strategy"] == strategy)
        & np.isclose(fit_df["phase_amplitude"], focus_amplitude)
    ]
    if fit_match.empty:
        continue
    fit = fit_match.iloc[0]
    fit_n = np.linspace(fit["n_min"], fit["n_max"], 300)
    ax.plot(
        fit_n,
        fit["intercept"] + fit["slope"] * fit_n,
        color=meta["color"],
        linestyle="--",
        linewidth=2.4,
        label="_nolegend_",
        zorder=3,
    )
    combined_handles.append(Line2D(
        [], [], color=meta["color"], linestyle="--", linewidth=2.4,
        marker=meta["marker"], markerfacecolor="white",
        markeredgecolor=meta["color"], markeredgewidth=1.7, markersize=7.2,
    ))
    combined_labels.append(
        rf"{meta['label']}: $\xi={scientific_latex(fit['slope'])}$"
    )

ax.set_xlabel(r"Number of shots $n$", fontsize=21)
ax.set_ylabel(r"$-\log\beta_n$", fontsize=21)
ax.set_xlim(0, 1.03 * CHECKPOINTS.max())
ax.set_ylim(0.0, 0.4)
focus_ticks = np.rint(np.linspace(0, CHECKPOINTS.max(), 5)).astype(int)
ax.set_xticks(focus_ticks)
ax.tick_params(axis="both", which="major", labelsize=18, direction="in", length=7, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=4, top=True, right=True)
ax.grid(False)
ax.legend(
    combined_handles, combined_labels, frameon=False, fontsize=14,
    ncol=1, loc="upper left", handlelength=2.8,
)
fig.tight_layout()
focus_amplitude_token = format(focus_amplitude, ".6g").replace(".", "p")
focus_path_pdf = FIGURE_DIR / f"{OUTPUT_STEM}_minus_log_beta_phi_{focus_amplitude_token}.pdf"
fig.savefig(focus_path_pdf, bbox_inches="tight")
plt.show()


In [ ]:
def bernoulli_kl(p0: np.ndarray | float, p1: np.ndarray | float) -> np.ndarray:
    p0 = np.clip(np.asarray(p0, dtype=float), 1.0e-15, 1.0 - 1.0e-15)
    p1 = np.clip(np.asarray(p1, dtype=float), 1.0e-15, 1.0 - 1.0e-15)
    return p0 * np.log(p0 / p1) + (1.0 - p0) * np.log((1.0 - p0) / (1.0 - p1))


readout = RabiReadoutConfig(
    contrast=float(actual_setup["contrast"]),
    readout_error=float(actual_setup["readout_error"]),
    t1=float(actual_setup["t1_time"]),
    t2=float(actual_setup["t2_time"]),
)
sensing_time = float(actual_setup["sensing_time"])
phase_samples = np.linspace(0.0, 2.0 * np.pi, 32768, endpoint=False)
EXPONENT_PLOT_PHASE_MIN = 0.05 if DETECTOR_PROFILE == "current" else 0.035
EXPONENT_PLOT_PHASE_MAX = 0.25
dense_theory_phase_grid = np.linspace(
    EXPONENT_PLOT_PHASE_MIN, EXPONENT_PLOT_PHASE_MAX, 500
)
LARGE_SIGNAL_PLOT_PHI_MIN = max(0.0, LARGE_SIGNAL_PHI_MIN - 0.01)
large_signal_theory_phase_grid = np.linspace(
    LARGE_SIGNAL_PHI_MIN, LARGE_SIGNAL_PHI_MAX, 500
)

p0_z = float(rabi_probability_one(0.0, 0.0, 0.0, 0.0, sensing_time, "z", 0.0, readout))
p0_transverse = float(rabi_probability_one(0.0, 0.0, 0.0, 0.0, sensing_time, "transverse", 0.0, readout))

kl_evaluation_grid = np.unique(np.concatenate([
    dense_theory_phase_grid,
    large_signal_theory_phase_grid,
    SELECTED_AMPLITUDES,
]))
kl_rows = []
for amplitude in kl_evaluation_grid:
    p1_z = float(rabi_probability_one(amplitude, 0.0, 0.0, 0.0, sensing_time, "z", 0.0, readout))
    rate_z = float(bernoulli_kl(p0_z, p1_z))

    p1_x = rabi_probability_one(amplitude, 0.0, phase_samples, 0.0, sensing_time, "transverse", 0.0, readout)
    p1_y = rabi_probability_one(amplitude, 0.0, phase_samples, 0.0, sensing_time, "transverse", 0.5 * np.pi, readout)
    rate_x_by_phase = bernoulli_kl(p0_transverse, p1_x)
    rate_xy_by_phase = 0.5 * (
        rate_x_by_phase + bernoulli_kl(p0_transverse, p1_y)
    )

    p1_oracle = float(
        rabi_probability_one(
            amplitude,
            0.0,
            0.0,
            0.0,
            sensing_time,
            "transverse",
            0.5 * np.pi,
            readout,
        )
    )
    rate_oracle = float(bernoulli_kl(p0_transverse, p1_oracle))
    kl_rows.append({
        "phase_amplitude": float(amplitude),
        "fixed_z": rate_z,
        "fixed_x_worst": float(np.min(rate_x_by_phase)),
        "fixed_x_phase_mean": float(np.mean(rate_x_by_phase)),
        "two_axis_worst": float(np.min(rate_xy_by_phase)),
        "two_axis_phase_mean": float(np.mean(rate_xy_by_phase)),
        "two_axis_best": float(np.max(rate_xy_by_phase)),
        "transverse_oracle": rate_oracle,
    })

kl_df = pd.DataFrame(kl_rows)
display(kl_df[kl_df["phase_amplitude"].isin(SELECTED_AMPLITUDES)])


def fixed_z_finite_log_beta(amplitude: float, n_shots: np.ndarray) -> np.ndarray:
    n_shots = np.asarray(n_shots, dtype=int)
    p1 = float(
        rabi_probability_one(amplitude, 0.0, 0.0, 0.0, sensing_time, "z", 0.0, readout)
    )
    # Reject for K > boundary and randomize at K = boundary so the test has exact size ALPHA.
    boundary = np.asarray(binom.isf(ALPHA, n_shots, p0_z), dtype=int)
    alpha_above = np.asarray(binom.sf(boundary, n_shots, p0_z), dtype=float)
    boundary_mass_h0 = np.asarray(binom.pmf(boundary, n_shots, p0_z), dtype=float)
    reject_probability_at_boundary = np.clip(
        (ALPHA - alpha_above) / boundary_mass_h0, 0.0, 1.0
    )
    beta = (
        binom.cdf(boundary - 1, n_shots, p1)
        + (1.0 - reject_probability_at_boundary) * binom.pmf(boundary, n_shots, p1)
    )
    return np.log(np.clip(np.asarray(beta, dtype=float), np.finfo(float).tiny, 1.0))


def two_axis_finite_log_beta(amplitude: float, n_shots: np.ndarray) -> np.ndarray:
    n_shots = np.asarray(n_shots, dtype=int)
    n_x = n_shots // 2
    n_y = n_shots - n_x
    p1_x = float(
        rabi_probability_one(amplitude, 0.0, 0.0, 0.0, sensing_time, "transverse", 0.0, readout)
    )
    p1_y = float(
        rabi_probability_one(
            amplitude, 0.0, 0.0, 0.0, sensing_time, "transverse", 0.5 * np.pi, readout
        )
    )
    variance_0 = p0_transverse * (1.0 - p0_transverse)
    noncentrality = (
        n_x * (p1_x - p0_transverse) ** 2
        + n_y * (p1_y - p0_transverse) ** 2
    ) / variance_0
    threshold = chi2.ppf(1.0 - ALPHA, df=2)
    return np.asarray(ncx2.logcdf(threshold, df=2, nc=noncentrality), dtype=float)


finite_theory_phase_grid = np.unique(np.concatenate([
    dense_theory_phase_grid,
    SELECTED_AMPLITUDES,
]))
finite_theory_endpoints = np.asarray([FINITE_N_THEORY_MIN, FINITE_N_THEORY_MAX], dtype=int)
finite_theory_rows = []
for strategy, log_beta_function, model in [
    ("fixed_z", fixed_z_finite_log_beta, "exact-size randomized binomial"),
    ("two_axis", two_axis_finite_log_beta, r"noncentral $\chi^2_2$"),
]:
    for amplitude in finite_theory_phase_grid:
        log_beta = log_beta_function(float(amplitude), finite_theory_endpoints)
        slope = float(-(log_beta[1] - log_beta[0]) / (FINITE_N_THEORY_MAX - FINITE_N_THEORY_MIN))
        finite_theory_rows.append({
            "strategy": strategy,
            "phase_amplitude": float(amplitude),
            "slope": slope,
            "n_min": FINITE_N_THEORY_MIN,
            "n_max": FINITE_N_THEORY_MAX,
            "model": model,
        })
fixed_finite_theory_df = pd.DataFrame(finite_theory_rows)
display(
    fixed_finite_theory_df[
        fixed_finite_theory_df["phase_amplitude"].isin(SELECTED_AMPLITUDES)
    ]
)


In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 6.2))

fixed_finite_theory_specs = [
    ("fixed_z", r"Fixed $z$ (finite-$n$)"),
    ("two_axis", r"Fixed $x/y$ (finite-$n$)"),
]
for strategy, label in fixed_finite_theory_specs:
    theory = fixed_finite_theory_df[
        fixed_finite_theory_df["strategy"] == strategy
    ].sort_values("phase_amplitude")
    ax.plot(
        theory["phase_amplitude"],
        theory["slope"],
        color=STYLE[strategy]["color"],
        linestyle="-",
        linewidth=2.5,
        label=label,
        zorder=4,
    )

endpoint_indices = [
    int(np.argmin(np.abs(CHECKPOINTS - FINITE_N_THEORY_MIN))),
    int(np.argmin(np.abs(CHECKPOINTS - FINITE_N_THEORY_MAX))),
]
if not np.array_equal(CHECKPOINTS[endpoint_indices], [FINITE_N_THEORY_MIN, FINITE_N_THEORY_MAX]):
    raise ValueError("Adaptive finite-n secant endpoints must be stored checkpoints")
adaptive_secant_rows = []
for strategy in ADAPTIVE_STRATEGIES:
    adaptive_endpoint_thresholds = np.asarray([
        null_threshold(LOG_B_H0[strategy][:, index], ALPHA)
        for index in endpoint_indices
    ])
    for phase_index, amplitude in enumerate(PHASE_GRID):
        samples = LOG_B_H1[strategy][phase_index][:, endpoint_indices]
        failures = np.sum(samples < adaptive_endpoint_thresholds[None, :], axis=0)
        total = int(samples.shape[0])
        beta_empirical = failures / total
        beta_center, beta_lower, beta_upper = beta_interval(failures, total, INTERVAL_LEVEL)
        delta_n = FINITE_N_THEORY_MAX - FINITE_N_THEORY_MIN
        slope = float(np.log(beta_center[0] / beta_center[1]) / delta_n)
        slope_lower = float(np.log(beta_lower[0] / beta_upper[1]) / delta_n)
        slope_upper = float(np.log(beta_upper[0] / beta_lower[1]) / delta_n)
        resolved = bool(np.all(failures >= MIN_FAILURES))
        adaptive_secant_rows.append({
            "strategy": strategy, "phase_amplitude": float(amplitude),
            "slope": slope, "slope_lower": slope_lower, "slope_upper": slope_upper,
            "beta_n_min": float(beta_empirical[0]), "beta_n_max": float(beta_empirical[1]),
            "failures_n_min": int(failures[0]), "failures_n_max": int(failures[1]),
            "resolved": resolved, "lower_limit": bool((not resolved) and slope_lower > 0.0),
        })
adaptive_finite_secant_df = pd.DataFrame(adaptive_secant_rows)
for strategy in PUBLICATION_ADAPTIVE_STRATEGIES:
    meta = STYLE[strategy]
    resolved_adaptive = adaptive_finite_secant_df[
        (adaptive_finite_secant_df["strategy"] == strategy)
        & adaptive_finite_secant_df["resolved"]
        & (adaptive_finite_secant_df["slope_lower"] > 0.0)
    ]
    if resolved_adaptive.empty:
        continue
    resolved_yerr = np.vstack([
        resolved_adaptive["slope"] - resolved_adaptive["slope_lower"],
        resolved_adaptive["slope_upper"] - resolved_adaptive["slope"],
    ])
    ax.errorbar(
        resolved_adaptive["phase_amplitude"], resolved_adaptive["slope"],
        yerr=resolved_yerr, fmt=meta["marker"], markerfacecolor="white",
        color=meta["color"], elinewidth=2.2, capsize=5.0, capthick=2.0,
        label=rf"{meta['label']} (finite-$n$)", zorder=5,
    )
# Retain these diagnostics for the saved summary, but keep them out of this comparison plot.
tail_mask = CHECKPOINTS >= max(FIT_N_MIN, CHECKPOINTS[len(CHECKPOINTS) // 2])
adaptive_fpol_tail = {
    strategy: float(np.mean(adaptive_fpol_h0[strategy][tail_mask]))
    for strategy in ADAPTIVE_STRATEGIES
}
reference_specs = [
    ("fixed_z", STYLE["fixed_z"]["color"], ":", r"Fixed $z$ (asymptotic)"),
    ("two_axis_worst", STYLE["two_axis"]["color"], ":", r"Fixed $x/y$ (asymptotic)"),
    ("transverse_oracle", "black", ":", "Transverse oracle (asymptotic)"),
]
for column, color, linestyle, label in reference_specs:
    ax.plot(
        kl_df["phase_amplitude"],
        kl_df[column],
        color=color,
        linestyle=linestyle,
        linewidth=1.8,
        label=label,
        zorder=2,
    )
ax.set_xlabel(r"Injected Rabi phase $\Phi_R$", fontsize=20)
ax.set_ylabel(r"$\xi_{\rm eff}$", fontsize=20)
ax.set_yscale("log")
ax.set_xlim(EXPONENT_PLOT_PHASE_MIN, EXPONENT_PLOT_PHASE_MAX)
ax.set_ylim(1.0e-6, 2.0e-2)
ax.tick_params(axis="both", which="major", labelsize=17, direction="in", length=7, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=4, top=True, right=True)
ax.grid(False, which="both")
legend_handles, legend_labels = ax.get_legend_handles_labels()
legend_by_label = dict(zip(legend_labels, legend_handles))
legend_order = [
    *[rf"{STYLE[strategy]['label']} (finite-$n$)" for strategy in PUBLICATION_ADAPTIVE_STRATEGIES],
    r"Fixed $x/y$ (finite-$n$)",
    r"Fixed $x/y$ (asymptotic)",
    r"Fixed $z$ (finite-$n$)",
    r"Fixed $z$ (asymptotic)",
    "Transverse oracle (asymptotic)",
]
ax.legend(
    [legend_by_label[label] for label in legend_order], legend_order,
    frameon=False, fontsize=12.5, ncol=1, loc="lower right", handlelength=2.5,
)
fig.tight_layout()
path_pdf = FIGURE_DIR / f"{OUTPUT_STEM}_relative_entropy_comparison.pdf"
fig.savefig(path_pdf, bbox_inches="tight")
plt.show()


In [ ]:
large_signal_endpoints = np.asarray([LARGE_SIGNAL_N_MIN, LARGE_SIGNAL_N_MAX], dtype=int)
large_fixed_rows = []
for strategy, log_beta_function, model in [
    ("fixed_z", fixed_z_finite_log_beta, "exact-size randomized binomial"),
    ("two_axis", two_axis_finite_log_beta, r"noncentral $\chi^2_2$"),
]:
    for amplitude in large_signal_theory_phase_grid:
        log_beta = log_beta_function(float(amplitude), large_signal_endpoints)
        slope = float(
            -(log_beta[1] - log_beta[0]) / (LARGE_SIGNAL_N_MAX - LARGE_SIGNAL_N_MIN)
        )
        large_fixed_rows.append({
            "strategy": strategy,
            "phase_amplitude": float(amplitude),
            "slope": slope,
            "n_min": LARGE_SIGNAL_N_MIN,
            "n_max": LARGE_SIGNAL_N_MAX,
            "model": model,
        })
large_fixed_theory_df = pd.DataFrame(large_fixed_rows)

large_endpoint_indices = [
    int(np.argmin(np.abs(CHECKPOINTS - LARGE_SIGNAL_N_MIN))),
    int(np.argmin(np.abs(CHECKPOINTS - LARGE_SIGNAL_N_MAX))),
]
if not np.array_equal(
    CHECKPOINTS[large_endpoint_indices], [LARGE_SIGNAL_N_MIN, LARGE_SIGNAL_N_MAX]
):
    raise ValueError("Large-signal secant endpoints must be stored checkpoints")
large_phase_indices = np.flatnonzero(
    (PHASE_GRID >= LARGE_SIGNAL_PHI_MIN - 1.0e-12)
    & (PHASE_GRID <= LARGE_SIGNAL_PHI_MAX + 1.0e-12)
)
large_adaptive_rows = []
for strategy in ADAPTIVE_STRATEGIES:
    large_thresholds = np.asarray([
        null_threshold(LOG_B_H0[strategy][:, index], ALPHA)
        for index in large_endpoint_indices
    ])
    for phase_index in large_phase_indices:
        amplitude = float(PHASE_GRID[phase_index])
        samples = LOG_B_H1[strategy][phase_index][:, large_endpoint_indices]
        failures = np.sum(samples < large_thresholds[None, :], axis=0)
        total = int(samples.shape[0])
        beta_center, beta_lower, beta_upper = beta_interval(failures, total, INTERVAL_LEVEL)
        delta_n = LARGE_SIGNAL_N_MAX - LARGE_SIGNAL_N_MIN
        slope = float(np.log(beta_center[0] / beta_center[1]) / delta_n)
        slope_lower = float(np.log(beta_lower[0] / beta_upper[1]) / delta_n)
        slope_upper = float(np.log(beta_upper[0] / beta_lower[1]) / delta_n)
        large_adaptive_rows.append({
            "strategy": strategy, "phase_amplitude": amplitude,
            "slope": slope, "slope_lower": slope_lower, "slope_upper": slope_upper,
            "failures_n_min": int(failures[0]), "failures_n_max": int(failures[1]),
            "resolved": bool(np.all(failures >= MIN_FAILURES)),
        })
large_adaptive_secant_df = pd.DataFrame(large_adaptive_rows)
display(large_adaptive_secant_df)

fig, ax = plt.subplots(figsize=(8.6, 6.2))
for strategy, label in [
    ("fixed_z", r"Fixed $z$ (finite-$n$)"),
    ("two_axis", r"Fixed $x/y$ (finite-$n$)"),
]:
    theory = large_fixed_theory_df[
        large_fixed_theory_df["strategy"] == strategy
    ].sort_values("phase_amplitude")
    ax.plot(
        theory["phase_amplitude"], theory["slope"],
        color=STYLE[strategy]["color"], linestyle="-", linewidth=2.5,
        label=label, zorder=4,
    )
for strategy in PUBLICATION_ADAPTIVE_STRATEGIES:
    meta = STYLE[strategy]
    large_resolved = large_adaptive_secant_df[
        (large_adaptive_secant_df["strategy"] == strategy)
        & large_adaptive_secant_df["resolved"]
        & (large_adaptive_secant_df["slope_lower"] > 0.0)
    ]
    if large_resolved.empty:
        continue
    large_yerr = np.vstack([
        large_resolved["slope"] - large_resolved["slope_lower"],
        large_resolved["slope_upper"] - large_resolved["slope"],
    ])
    ax.errorbar(
        large_resolved["phase_amplitude"], large_resolved["slope"],
        yerr=large_yerr, fmt=meta["marker"], markerfacecolor="white",
        color=meta["color"], elinewidth=2.2, capsize=5.0, capthick=2.0,
        label=rf"{meta['label']} (finite-$n$)", zorder=5,
    )
for column, color, label in [
    ("fixed_z", STYLE["fixed_z"]["color"], r"Fixed $z$ (asymptotic)"),
    ("two_axis_worst", STYLE["two_axis"]["color"], r"Fixed $x/y$ (asymptotic)"),
    ("transverse_oracle", "black", "Transverse oracle (asymptotic)"),
]:
    ax.plot(
        kl_df["phase_amplitude"], kl_df[column], color=color, linestyle=":",
        linewidth=1.8, label=label, zorder=2,
    )
ax.set_xlabel(r"Injected Rabi phase $\Phi_R$", fontsize=20)
ax.set_ylabel(r"Type-II exponent $\xi$", fontsize=20)
ax.set_yscale("log")
ax.set_xlim(LARGE_SIGNAL_PLOT_PHI_MIN, LARGE_SIGNAL_PHI_MAX + 0.01)
ax.set_ylim(5.0e-6, 3.0e-2)
ax.tick_params(axis="both", which="major", labelsize=17, direction="in", length=7, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=4, top=True, right=True)
ax.grid(False, which="both")
legend_handles, legend_labels = ax.get_legend_handles_labels()
legend_by_label = dict(zip(legend_labels, legend_handles))
legend_order = [
    *[rf"{STYLE[strategy]['label']} (finite-$n$)" for strategy in PUBLICATION_ADAPTIVE_STRATEGIES],
    r"Fixed $x/y$ (finite-$n$)",
    r"Fixed $x/y$ (asymptotic)",
    r"Fixed $z$ (finite-$n$)",
    r"Fixed $z$ (asymptotic)",
    "Transverse oracle (asymptotic)",
]
ax.legend(
    [legend_by_label[label] for label in legend_order], legend_order,
    frameon=False, fontsize=12.5, ncol=1, loc="lower right", handlelength=2.5,
)
fig.tight_layout()
large_path_pdf = FIGURE_DIR / f"{OUTPUT_STEM}_relative_entropy_comparison_large_signal.pdf"
fig.savefig(large_path_pdf, bbox_inches="tight")
plt.show()

large_fixed_path = FIGURE_DIR / f"{OUTPUT_STEM}_fixed_large_signal_finite_n_theory.csv"
large_adaptive_path = FIGURE_DIR / f"{OUTPUT_STEM}_adaptive_large_signal_finite_n_secant.csv"
large_fixed_theory_df.to_csv(large_fixed_path, index=False)
large_adaptive_secant_df.to_csv(large_adaptive_path, index=False)
print(large_path_pdf)
print(large_fixed_path)
print(large_adaptive_path)


In [ ]:
diagnostic_rows = []
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))
for strategy in ["two_axis", *ADAPTIVE_STRATEGIES]:
    if strategy not in STRATEGIES:
        continue
    meta = STYLE[strategy]
    mean_a = np.mean(PREDICTABLE_KL_H0[strategy], axis=0)
    mean_v = np.mean(PREDICTABLE_VARIANCE_H0[strategy], axis=0)
    observed = -np.mean(LOG_B_H0[strategy], axis=0)
    finite_rate = (
        mean_a + normal_distribution.ppf(ALPHA) * np.sqrt(np.maximum(mean_v, 0.0))
    ) / CHECKPOINTS
    axes[0].plot(
        CHECKPOINTS,
        observed,
        marker=meta["marker"],
        markerfacecolor="white",
        markersize=4.5,
        color=meta["color"],
        label=rf"{meta['label']}: $-\langle\log B_{{10}}\rangle_0$",
    )
    axes[0].plot(
        CHECKPOINTS,
        mean_a,
        linestyle="--",
        linewidth=1.7,
        color=meta["color"],
        label=rf"{meta['label']}: $\langle A_n\rangle_0$",
    )
    axes[1].plot(
        CHECKPOINTS,
        mean_a / CHECKPOINTS,
        linewidth=1.8,
        color=meta["color"],
        label=rf"{meta['label']}: $A_n/n$",
    )
    axes[1].plot(
        CHECKPOINTS,
        finite_rate,
        linestyle=":",
        linewidth=1.8,
        color=meta["color"],
        label=rf"{meta['label']}: $(A_n+z_\alpha\sqrt{{V_n}})/n$",
    )
    for checkpoint_index, n_shots in enumerate(CHECKPOINTS):
        diagnostic_rows.append({
            "strategy": strategy,
            "n_shots": int(n_shots),
            "mean_predictable_kl": float(mean_a[checkpoint_index]),
            "mean_predictable_variance": float(mean_v[checkpoint_index]),
            "minus_mean_log_bayes_h0": float(observed[checkpoint_index]),
            "mixture_kl_rate": float(mean_a[checkpoint_index] / n_shots),
            "finite_n_mixture_rate": float(finite_rate[checkpoint_index]),
        })

for ax in axes:
    ax.set_xscale("log", base=2)
    ax.set_xlabel(r"Number of shots $n_{\rm shot}$", fontsize=15)
    ax.tick_params(axis="both", which="major", labelsize=12, direction="in", top=True, right=True)
    ax.grid(False)
axes[0].set_ylabel("Cumulative information", fontsize=15)
axes[1].set_ylabel("Information per shot", fontsize=15)
axes[0].legend(frameon=False, fontsize=9)
axes[1].legend(frameon=False, fontsize=9)
fig.tight_layout()
diagnostic_pdf_path = FIGURE_DIR / f"{OUTPUT_STEM}_predictable_kl_diagnostic.pdf"
fig.savefig(diagnostic_pdf_path, bbox_inches="tight")
plt.show()

diagnostic_df = pd.DataFrame(diagnostic_rows)
fpol_rows = [
    {
        "strategy": strategy,
        "n_shots": int(n_shots),
        "f_pol_h0_phase_mean": float(adaptive_fpol_h0[strategy][index]),
    }
    for strategy in ADAPTIVE_STRATEGIES
    for index, n_shots in enumerate(CHECKPOINTS)
]
fpol_df = pd.DataFrame(fpol_rows)
display(diagnostic_df.tail())
display(fpol_df.tail())


In [ ]:
summary = {
    "raw_cache": str(RAW_CACHE_PATH),
    "raw_setup_hash": RAW_SETUP_HASH,
    "alpha": ALPHA,
    "selected_amplitudes": SELECTED_AMPLITUDES.tolist(),
    "fit_n_min": FIT_N_MIN,
    "fit_beta_max": FIT_BETA_MAX,
    "min_failures": MIN_FAILURES,
    "adaptive_f_pol_tail": adaptive_fpol_tail,
    "finite_n_theory_range": [FINITE_N_THEORY_MIN, FINITE_N_THEORY_MAX],
    "large_signal_range": {
        "n": [LARGE_SIGNAL_N_MIN, LARGE_SIGNAL_N_MAX],
        "phase": [LARGE_SIGNAL_PHI_MIN, LARGE_SIGNAL_PHI_MAX],
    },
    "notes": {
        "fixed_x": "The phase-averaged alternative contains blind phases, so the worst-case Stein exponent is zero.",
        "adaptive": "The cache directly stores Q_n, A_n, and V_n; no angle-history rerun or fit-inferred f_pol is used.",
    },
}
summary_path = FIGURE_DIR / f"{OUTPUT_STEM}_summary.json"
beta_path = FIGURE_DIR / f"{OUTPUT_STEM}_beta.csv"
fit_path = FIGURE_DIR / f"{OUTPUT_STEM}_fits.csv"
corrected_fit_path = FIGURE_DIR / f"{OUTPUT_STEM}_adaptive_second_order_fits.csv"
kl_path = FIGURE_DIR / f"{OUTPUT_STEM}_relative_entropy.csv"
fixed_finite_theory_path = FIGURE_DIR / f"{OUTPUT_STEM}_fixed_finite_n_theory.csv"
adaptive_finite_secant_path = FIGURE_DIR / f"{OUTPUT_STEM}_adaptive_finite_n_secant.csv"
diagnostic_path = FIGURE_DIR / f"{OUTPUT_STEM}_predictable_kl_diagnostic.csv"
fpol_path = FIGURE_DIR / f"{OUTPUT_STEM}_adaptive_f_pol.csv"
summary_path.write_text(json.dumps(summary, indent=2))
beta_df.to_csv(beta_path, index=False)
fit_df.to_csv(fit_path, index=False)
adaptive_corrected_fit_df.to_csv(corrected_fit_path, index=False)
kl_df.to_csv(kl_path, index=False)
fixed_finite_theory_df.to_csv(fixed_finite_theory_path, index=False)
adaptive_finite_secant_df.to_csv(adaptive_finite_secant_path, index=False)
diagnostic_df.to_csv(diagnostic_path, index=False)
fpol_df.to_csv(fpol_path, index=False)
print(summary_path)
print(beta_path)
print(fit_path)
print(corrected_fit_path)
print(kl_path)
print(fixed_finite_theory_path)
print(adaptive_finite_secant_path)
print(diagnostic_path)
print(fpol_path)
